In [6]:
!pip -q install kagglehub

import kagglehub
import pandas as pd
import os
import glob

# Download the IBM Telco Customer Churn dataset
path = kagglehub.dataset_download(
    "blastchar/telco-customer-churn"
)

print("Dataset path:", path)

# Find the CSV file
csv_files = glob.glob(
    os.path.join(path, "**", "*.csv"),
    recursive=True
)

for file in csv_files:
    print("\nFile:", file)

    preview = pd.read_csv(file, nrows=5)

    print("Columns:")
    print(preview.columns.tolist())

    print("\nPreview:")
    display(preview.head())

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Dataset path: /kaggle/input/telco-customer-churn

File: /kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv
Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Preview:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Data source:
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn

# Load the dataset
file_path = (
    "/kaggle/input/telco-customer-churn/"
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(file_path)

# Convert TotalCharges from text to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Convert Churn into numbers
# No = 0, Yes = 1
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Select the features for the model
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "InternetService",
    "Contract",
    "TechSupport",
    "PaymentMethod"
]

selected_columns = (
    numeric_features +
    categorical_features +
    ["Churn"]
)

df_model = df[selected_columns].dropna()

# Features and target
X = df_model[numeric_features + categorical_features]
y = df_model["Churn"]

print("Number of records used:", len(df_model))
print("Number of churned customers:", y.sum())

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

# Create the logistic regression pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            random_state=42,
            max_iter=1000
        )
    )
])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train the model
model.fit(X_train, y_train)

# Create a new customer for prediction
new_customer = pd.DataFrame({
    "SeniorCitizen": [0],
    "tenure": [12],
    "MonthlyCharges": [75.00],
    "TotalCharges": [900.00],
    "InternetService": ["Fiber optic"],
    "Contract": ["Month-to-month"],
    "TechSupport": ["No"],
    "PaymentMethod": ["Electronic check"]
})

# Predict the probability of churn
churn_probability = model.predict_proba(
    new_customer
)[0][1]

# Classify using a 0.5 threshold
threshold = 0.5

churn_prediction = (
    1 if churn_probability >= threshold else 0
)

print(
    f"\nChurn probability for the new customer: "
    f"{churn_probability:.2%}"
)

print(
    f"Churn prediction "
    f"(1 = churn, 0 = no churn): {churn_prediction}"
)

# Display model coefficients
feature_names = (
    model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    model.named_steps["classifier"]
    .coef_[0]
)

print("\nModel Coefficients:")

for feature, coefficient in zip(
    feature_names,
    coefficients
):
    print(f"{feature}: {coefficient:.4f}")

Number of records used: 7032
Number of churned customers: 1869

Churn probability for the new customer: 63.54%
Churn prediction (1 = churn, 0 = no churn): 1

Model Coefficients:
num__SeniorCitizen: 0.1177
num__tenure: -1.3682
num__MonthlyCharges: 0.0365
num__TotalCharges: 0.6966
cat__InternetService_Fiber optic: 0.7537
cat__InternetService_No: -0.4606
cat__Contract_One year: -0.8033
cat__Contract_Two year: -1.4725
cat__TechSupport_No internet service: -0.4606
cat__TechSupport_Yes: -0.4286
cat__PaymentMethod_Credit card (automatic): 0.0553
cat__PaymentMethod_Electronic check: 0.4769
cat__PaymentMethod_Mailed check: 0.0228


The Churn probabilty shows the likely hood a customer will stop doing business for said company. Businesses can use Churn probability to keep customers.